## Joining tables with weather per city and population 

In [43]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
from matplotlib.dates import date2num
import pandas as pd

In [34]:
df_pop = pd.read_csv('./data/England_city_population_monthly_21_25.csv')
df_w = pd.read_csv('./data/all_weather_clean.csv')
df_p = pd.read_csv('./data/England_pharm_per_city_21_25.csv')

In [37]:
df_w = df_w.rename(columns={
    'station_name': 'city'
})
df_pop['city'] = df_pop['city'].replace('Brighton and Hove', 'Brighton')

In [38]:
df_w_pop = df_w.merge(df_pop,  on=['date', 'city'], how='left')

In [39]:
england_cities = [
    'Sunderland', 'Newcastle upon Tyne', 'Leeds', 'Bristol', 'London',
    'Middlesbrough', 'Manchester', 'York', 'Nottingham', 'Peterborough',
    'Birmingham', 'Plymouth', 'Exeter', 'Brighton', 'Canterbury'
]

df_w_pop = df_w_pop[df_w_pop['city'].isin(england_cities)]
df_w_pop.tail()

,date,tavg,tmin,tmax,prcp,pres,tsun,city,population
1375,2021-07,16.9,13.9,19.8,NaN,1014.2,8138.0,Sunderland,276253.0
1376,2022-01,6.5,4.3,8.5,NaN,1020.8,3962.0,Sunderland,277000.0
1377,2022-02,7.5,5.4,9.6,NaN,1006.8,3412.0,Sunderland,277125.0
1378,2022-03,7.7,4.8,11.0,NaN,1022.8,5041.0,Sunderland,277250.0
1379,2022-04,8.5,5.7,11.6,NaN,1016.3,7932.0,Sunderland,277375.0


In [40]:
df_w_pop.to_csv("England_city_weather_population.csv", index=False)

In [41]:
df_all_cities = df_p.merge(df_w_pop,on=['date', 'city'], how='left')
df_all_cities.head()

,date,city,chemical_substance,group,items,actual_cost,tavg,tmin,tmax,prcp,pres,tsun,population
0,2021-01,Birmingham,Agomelatine,antidepressant,10.0,477.1802,3.3,0.8,5.5,56.0,1010.2,4042.0,1144900.0
1,2021-01,Birmingham,Amitriptyline hydrochloride,antidepressant,18253.0,37451.0438,3.3,0.8,5.5,56.0,1010.2,4042.0,1144900.0
2,2021-01,Birmingham,Amobarbital sodium,anxiolytic,1.0,216.8842,3.3,0.8,5.5,56.0,1010.2,4042.0,1144900.0
3,2021-01,Birmingham,Bupropion hydrochloride,antidepressant,12.0,577.3022,3.3,0.8,5.5,56.0,1010.2,4042.0,1144900.0
4,2021-01,Birmingham,Buspirone hydrochloride,anxiolytic,214.0,3823.5488,3.3,0.8,5.5,56.0,1010.2,4042.0,1144900.0


In [45]:
df_all_cities['date'] = pd.to_datetime(df_all_cities['date'])
# add days in month
df_all_cities['days_in_month'] = df_all_cities['date'].dt.days_in_month

# normalize per 1000 people per day
df_all_cities['items_per_1k_per_day'] = round((df_all_cities['items'] /(df_all_cities['population'] * df_all_cities['days_in_month'])) * 1000,2)

In [46]:
df_all_cities

,date,city,chemical_substance,group,items,actual_cost,tavg,tmin,tmax,prcp,pres,tsun,population,days_in_month,items_per_1k_per_day
0,2021-01-01,Birmingham,Agomelatine,antidepressant,10.0,477.1802,3.3,0.8,5.5,56.0,1010.2,4042.0,1144900.0,31,0.00
1,2021-01-01,Birmingham,Amitriptyline hydrochloride,antidepressant,18253.0,37451.0438,3.3,0.8,5.5,56.0,1010.2,4042.0,1144900.0,31,0.51
2,2021-01-01,Birmingham,Amobarbital sodium,anxiolytic,1.0,216.8842,3.3,0.8,5.5,56.0,1010.2,4042.0,1144900.0,31,0.00
3,2021-01-01,Birmingham,Bupropion hydrochloride,antidepressant,12.0,577.3022,3.3,0.8,5.5,56.0,1010.2,4042.0,1144900.0,31,0.00
4,2021-01-01,Birmingham,Buspirone hydrochloride,anxiolytic,214.0,3823.5488,3.3,0.8,5.5,56.0,1010.2,4042.0,1144900.0,31,0.01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45346,2025-12-01,York,Trimipramine maleate,antidepressant,2.0,410.9048,6.8,4.9,8.6,92.0,1013.1,2471.0,212926.0,31,0.00
45347,2025-12-01,York,Venlafaxine,antidepressant,1752.0,14123.5275,6.8,4.9,8.6,92.0,1013.1,2471.0,212926.0,31,0.27
45348,2025-12-01,York,Vortioxetine,antidepressant,204.0,4889.7833,6.8,4.9,8.6,92.0,1013.1,2471.0,212926.0,31,0.03
45349,2025-12-01,York,Zolpidem tartrate,anxiolytic,74.0,48.3280,6.8,4.9,8.6,92.0,1013.1,2471.0,212926.0,31,0.01


In [47]:
df_all_cities.to_csv("England_all_per_city.csv", index=False)